# Notebook for exploatory data analysis
Included checks:
- [] max and min transaction timestamps for fetched collections to make sure all records are properly extracted

## Imports & variable assignment

In [28]:
import pandas as pd
from datetime import datetime, timezone
import glob
import json
import os

In [35]:
opensea_raw_data_folder = "data/raw2/opensea/"

## Load data to dataframes

In [36]:
def load_json_folder(folder_path, collection_slug):
    """
    Loads all JSON files from a folder located one directory above the current working directory.
    Combines folder_path and collection_slug to form the full path.
    """
    
    # Assemble base path
    base_dir = str(os.path.abspath(os.path.join(os.getcwd(), "..")))
    full_path = os.path.join(base_dir, folder_path, collection_slug)
    
    # Match all JSON files
    json_files = glob.glob(os.path.join(full_path, "*.json"))
    print(f"✅ Found {len(json_files)} JSON files in {folder_path}{collection_slug}")
    
    data_list = []
    for file in json_files:
        with open(file, 'r', encoding='utf-8') as f:
            data = json.load(f)
            if isinstance(data, list):
                data_list.extend(data)
            else:
                data_list.append(data)
    
    df = pd.DataFrame(data_list)
    print(f"📊 DataFrame created with {len(df)} rows and {len(df.columns)} columns from {folder_path}{collection_slug}")
    return df

In [37]:
bored_ape_data_df = load_json_folder(folder_path=opensea_raw_data_folder, collection_slug="boredapeyachtclub")
cryptopunks_data_df = load_json_folder(folder_path=opensea_raw_data_folder, collection_slug="cryptopunks")
pudgypenguins_data_df = load_json_folder(folder_path=opensea_raw_data_folder, collection_slug="pudgypenguins")

✅ Found 29 JSON files in data/raw2/opensea/boredapeyachtclub
📊 DataFrame created with 1401 rows and 12 columns from data/raw2/opensea/boredapeyachtclub
✅ Found 13 JSON files in data/raw2/opensea/cryptopunks
📊 DataFrame created with 649 rows and 10 columns from data/raw2/opensea/cryptopunks
✅ Found 47 JSON files in data/raw2/opensea/pudgypenguins
📊 DataFrame created with 2349 rows and 12 columns from data/raw2/opensea/pudgypenguins


## Check timestamps

In [24]:
def get_min_max_timestamp(df, timestamp_col='event_timestamp'):
    """
    Returns the minimum and maximum timestamp from a DataFrame column.
    Converts Unix timestamps to human-readable datetime.
    """
    if timestamp_col not in df.columns or df.empty:
        return None, None
    
    min_ts = df[timestamp_col].min()
    max_ts = df[timestamp_col].max()
    
    # Convert to readable datetime
    min_dt = datetime.fromtimestamp(min_ts, tz=timezone.utc)
    max_dt = datetime.fromtimestamp(max_ts, tz=timezone.utc)
    
    return min_dt, max_dt

In [25]:
min_dt, max_dt = get_min_max_timestamp(bored_ape_data_df)
print(f"⏰ Min - max timestamp: {min_dt} → {max_dt}")

Min - max timestamp: 2025-06-30 23:15:11+00:00 → 2025-09-29 20:29:47+00:00


In [26]:
min_dt, max_dt = get_min_max_timestamp(cryptopunks_data_df)
print(f"⏰ Min - max timestamp: {min_dt} → {max_dt}")

⏰ Min - max timestamp: 2025-07-01 04:55:35+00:00 → 2025-09-29 17:38:47+00:00


In [27]:
min_dt, max_dt = get_min_max_timestamp(pudgypenguins_data_df)
print(f"⏰ Min - max timestamp: {min_dt} → {max_dt}")

⏰ Min - max timestamp: 2025-07-01 07:13:23+00:00 → 2025-09-29 20:53:11+00:00


## Explore data structure
Explore schema on a sample df

In [38]:
bored_ape_data_df.head()

,event_type,event_timestamp,transaction,chain,payment,closing_date,seller,buyer,quantity,nft,order_hash,protocol_address
0,sale,1756145615,0x4eea299e3398905dac23b6f8e6f98d47a5344459a308...,ethereum,"{'quantity': '12440000000000000000', 'token_ad...",1756145615,0x0000000188e3604489698ea73de28524f2bea6c6,0xa66851a76c6d3e00ad49c4fd00cc4c1444962cbc,1,"{'identifier': '8342', 'collection': 'boredape...",NaN,NaN
1,sale,1756144523,0x397d9303e78dbf9f6848f5fcd750ea81fd6dacbca74d...,ethereum,"{'quantity': '10299999900000000000', 'token_ad...",1756144523,0xf76246b0842c92ad5bd745973ca9eb85b937b126,0x4b3e77089c5eb6558b5355eab3bf2ee4156bf2a9,1,"{'identifier': '6999', 'collection': 'boredape...",NaN,NaN
2,sale,1756142195,0x7ffca3ef9d8d178a646c961b5ee96d55ef3b9cdfd993...,ethereum,"{'quantity': '10300000000000000000', 'token_ad...",1756142195,0xd360b4cbdd34cd4ea70806f45aed848d92843a5c,0xf2a7be2d334db950b8e490def5412827eac8494c,1,"{'identifier': '8144', 'collection': 'boredape...",NaN,NaN
3,sale,1756139927,0xfa3fa52912c86d9b6db7e56fcb76b24417fccf21944a...,ethereum,"{'quantity': '9880000000000000000', 'token_add...",1756139927,0x18196ccbaa2c9ac038a20ac541b6804e1829c926,0xd360b4cbdd34cd4ea70806f45aed848d92843a5c,1,"{'identifier': '8144', 'collection': 'boredape...",NaN,NaN
4,sale,1756137443,0xe59929ef31a614fb3d86c1aeb53dc7675a4606440f5d...,ethereum,"{'quantity': '13500000000000000000', 'token_ad...",1756137443,0x2924639e8af1f35a08c34334b387bc6dc057db8b,0x9807a9b6f6067ab89b7ae792f074f4fdf1923c3c,1,"{'identifier': '3609', 'collection': 'boredape...",NaN,NaN


In [56]:
# Explore main columns
for column in bored_ape_data_df.columns:
    print(column)

event_type
event_timestamp
transaction
chain
payment
closing_date
seller
buyer
quantity
nft
order_hash
protocol_address


In [59]:
# Explore nested payment columns
nested_columns = bored_ape_data_df['payment'].iloc[0].keys()
for col in nested_columns:
    print(col)

quantity
token_address
decimals
symbol


In [60]:
# Explore nested nft columns
nested_columns = bored_ape_data_df['nft'].iloc[0].keys()
for col in nested_columns:
    print(col)

identifier
collection
contract
token_standard
name
description
image_url
display_image_url
display_animation_url
metadata_url
opensea_url
updated_at
is_disabled
is_nsfw
original_image_url
original_animation_url


## Columns needed for further analysis

**Participants & identity**
- seller
- buyer
-> Detects:
Self‑trades
Wallet pairs trading repeatedly
Wallet clusters (same actor, multiple wallets)

**Time & sequencing**
- event_timestamp
- event_date
- closing_date
Detects:
Rapid buy‑sell cycles
Unrealistically short holding periods
Time‑based anomalies

**Asset identity**
-nft.token_address
-nft.collection
-nft.contract
-nft.identifier (from nft → symbol / identifier)
Detects:
Repeated trading of the same NFT
Single‑NFT volume inflation
Intra‑collection wash patterns

**Trade value**
-payment.quantity
-payment.quantity.protocol_address
-payment.decimals
-payment.symbol
Detects:
Artificial price pumping
Trades priced just above marketplace fees
Circular ETH/WETH movement

**Transaction linkage**
-transaction
-order_hash
-chain
Detects:
Linked orders
Cross‑order manipulation
Multi‑trade batching in one transaction

In [52]:
df = pd.read_parquet("/Users/juliaprzygoda/Desktop/thesis-code/nft-wash-trading-detection/data/processed/opensea/boredapeyachtclub_sales.parquet")
df.head()

,event_type,event_timestamp,event_day,chain,token_amount,is_bundle
